# Project 3: GPU Image Processing Pipeline
**Gaussian blur + Sobel edge detection — GPU vs CPU benchmark**

**Six kernels built:**
1. `rgb_to_grayscale` — parallel pixel conversion using ITU-R BT.601 weights
2. `gaussian_blur_naive` — 2D convolution, global memory only (baseline)
3. `gaussian_blur_horiz/vert` — separable 1D filter (5× fewer FLOPs than 2D)
4. `gaussian_blur_shmem` — tiled 2D with shared memory + halo regions
5. `sobel_edge_detect` — Gx/Gy gradient magnitude + binary threshold
6. `sobel_edge_shmem` — Sobel with shared memory tile loading

**Target:** 30–50× speedup over CPU for the full pipeline on 4K images.

**Why this project:** 2D image convolution is the core operation of CNN forward pass.
Building it from scratch proves you understand GPU memory hierarchy at the hardware level.

In [ ]:
# ── Cell 1: Verify GPU ──────────────────────────────────────────────
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!nvcc --version | grep release
import torch
print(f'PyTorch: {torch.__version__}  CUDA: {torch.version.cuda}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Shared mem/SM: {torch.cuda.get_device_properties(0).shared_memory_per_block/1024:.0f} KB')

In [ ]:
# ── Cell 2: Install dependencies ────────────────────────────────────
!pip install Pillow opencv-python-headless numpy matplotlib -q
print('Dependencies ready')

In [ ]:
# ── Cell 3: Download a real test image (4K) ─────────────────────────
import urllib.request, os

# Download a free 4K nature photo from Unsplash (CC0)
url = 'https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=3840&q=80'
img_path = 'test_4k.jpg'

if not os.path.exists(img_path):
    print('Downloading 4K test image...')
    try:
        urllib.request.urlretrieve(url, img_path)
        print(f'Downloaded: {os.path.getsize(img_path)/1e6:.1f} MB')
    except Exception as e:
        print(f'Download failed ({e}), generating synthetic image...')
        import numpy as np
        from PIL import Image
        # Generate a rich synthetic 4K test image
        W, H = 3840, 2160
        img = np.zeros((H, W, 3), dtype=np.uint8)
        # Gradient background
        for y in range(H):
            img[y, :, 0] = int(255 * y / H)
            img[y, :, 1] = int(180 * (1 - y/H))
        # Checkerboard for edges
        for y in range(0, H, 64):
            for x in range(0, W, 64):
                if (y//64 + x//64) % 2 == 0:
                    img[y:y+64, x:x+64, 2] = 200
        # Circles
        import math
        for r in range(50, 400, 80):
            for theta in range(360):
                cx = int(W//2 + r * math.cos(math.radians(theta)))
                cy = int(H//2 + r * math.sin(math.radians(theta)))
                if 0<=cx<W and 0<=cy<H:
                    img[cy, cx, :] = 255
        Image.fromarray(img).save(img_path, quality=85)
        print(f'Generated synthetic 4K image: {os.path.getsize(img_path)/1e6:.1f} MB')
else:
    print(f'Using cached image: {os.path.getsize(img_path)/1e6:.1f} MB')

# Load and inspect
from PIL import Image
import numpy as np
img_pil = Image.open(img_path).convert('RGB')
print(f'Image size: {img_pil.size[0]}×{img_pil.size[1]}  ({img_pil.size[0]*img_pil.size[1]/1e6:.1f} MP)')

In [ ]:
# ── Cell 4: Write all CUDA source files ─────────────────────────────
import os
os.makedirs('img_pipeline', exist_ok=True)
os.chdir('img_pipeline')

cuda_code = r'''
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#include <float.h>
#include <time.h>
#include <cuda_runtime.h>

#define BLOCK_W   32
#define BLOCK_H    8
#define KERNEL_R   2
#define KERNEL_SIZE (2*KERNEL_R+1)
#define HALO        KERNEL_R
#define SHMEM_W    (BLOCK_W + 2*HALO)
#define SHMEM_H    (BLOCK_H + 2*HALO)
#define SOBEL_BW   32
#define SOBEL_BH    8
#define SOBEL_SW   (SOBEL_BW + 2)
#define SOBEL_SH   (SOBEL_BH + 2)

#define CUDA_CHECK(x) do { cudaError_t e=(x); \
    if(e!=cudaSuccess){fprintf(stderr,"CUDA %s:%d %s\n",__FILE__,__LINE__,cudaGetErrorString(e));exit(1);} } while(0)

/* ── Constant memory ─────────────────────────────────────────────── */
__constant__ float c_gauss2d[KERNEL_SIZE][KERNEL_SIZE] = {
    {0.00296902f,0.01330621f,0.02193823f,0.01330621f,0.00296902f},
    {0.01330621f,0.05963430f,0.09832033f,0.05963430f,0.01330621f},
    {0.02193823f,0.09832033f,0.16210282f,0.09832033f,0.02193823f},
    {0.01330621f,0.05963430f,0.09832033f,0.05963430f,0.01330621f},
    {0.00296902f,0.01330621f,0.02193823f,0.01330621f,0.00296902f}
};
__constant__ float c_gauss1d[KERNEL_SIZE] = {0.0625f,0.25f,0.375f,0.25f,0.0625f};

/* ── KERNEL 1: RGB → Grayscale ───────────────────────────────────── */
__global__ void rgb_to_grayscale(const unsigned char* __restrict__ rgb,
    float* __restrict__ gray, int W, int H) {
    int x=blockIdx.x*blockDim.x+threadIdx.x, y=blockIdx.y*blockDim.y+threadIdx.y;
    if(x>=W||y>=H) return;
    int px=(y*W+x)*3;
    gray[y*W+x]=0.299f*(float)rgb[px]+0.587f*(float)rgb[px+1]+0.114f*(float)rgb[px+2];
}

/* ── KERNEL 2: Naive 2D Gaussian ─────────────────────────────────── */
__global__ void gaussian_blur_naive(const float* __restrict__ in,
    float* __restrict__ out, int W, int H) {
    int x=blockIdx.x*blockDim.x+threadIdx.x, y=blockIdx.y*blockDim.y+threadIdx.y;
    if(x>=W||y>=H) return;
    float acc=0.f;
    for(int ky=-KERNEL_R;ky<=KERNEL_R;++ky)
        for(int kx=-KERNEL_R;kx<=KERNEL_R;++kx)
            acc+=c_gauss2d[ky+KERNEL_R][kx+KERNEL_R]*
                 in[min(max(y+ky,0),H-1)*W+min(max(x+kx,0),W-1)];
    out[y*W+x]=acc;
}

/* ── KERNEL 3a: Separable Horizontal ─────────────────────────────── */
__global__ void gaussian_blur_horiz(const float* __restrict__ in,
    float* __restrict__ out, int W, int H) {
    int x=blockIdx.x*blockDim.x+threadIdx.x, y=blockIdx.y*blockDim.y+threadIdx.y;
    if(x>=W||y>=H) return;
    float acc=0.f;
    #pragma unroll
    for(int kx=-KERNEL_R;kx<=KERNEL_R;++kx)
        acc+=c_gauss1d[kx+KERNEL_R]*in[y*W+min(max(x+kx,0),W-1)];
    out[y*W+x]=acc;
}

/* ── KERNEL 3b: Separable Vertical ───────────────────────────────── */
__global__ void gaussian_blur_vert(const float* __restrict__ in,
    float* __restrict__ out, int W, int H) {
    int x=blockIdx.x*blockDim.x+threadIdx.x, y=blockIdx.y*blockDim.y+threadIdx.y;
    if(x>=W||y>=H) return;
    float acc=0.f;
    #pragma unroll
    for(int ky=-KERNEL_R;ky<=KERNEL_R;++ky)
        acc+=c_gauss1d[ky+KERNEL_R]*in[min(max(y+ky,0),H-1)*W+x];
    out[y*W+x]=acc;
}

/* ── KERNEL 4: Shared memory tiled Gaussian ──────────────────────── */
__global__ void gaussian_blur_shmem(const float* __restrict__ in,
    float* __restrict__ out, int W, int H) {
    __shared__ float smem[SHMEM_H][SHMEM_W];
    int tx=threadIdx.x,ty=threadIdx.y;
    int x=blockIdx.x*BLOCK_W+tx, y=blockIdx.y*BLOCK_H+ty;
    for(int dy=ty;dy<SHMEM_H;dy+=BLOCK_H)
        for(int dx=tx;dx<SHMEM_W;dx+=BLOCK_W) {
            int gx=blockIdx.x*BLOCK_W+dx-HALO, gy=blockIdx.y*BLOCK_H+dy-HALO;
            smem[dy][dx]=in[min(max(gy,0),H-1)*W+min(max(gx,0),W-1)];
        }
    __syncthreads();
    if(x>=W||y>=H) return;
    float acc=0.f;
    #pragma unroll
    for(int ky=0;ky<KERNEL_SIZE;++ky)
        #pragma unroll
        for(int kx=0;kx<KERNEL_SIZE;++kx)
            acc+=c_gauss2d[ky][kx]*smem[ty+ky][tx+kx];
    out[y*W+x]=acc;
}

/* ── KERNEL 5: Sobel with shared memory ──────────────────────────── */
__global__ void sobel_edge_shmem(const float* __restrict__ in,
    float* __restrict__ edges, unsigned char* __restrict__ edge_u8,
    int W, int H, float thr) {
    __shared__ float smem[SOBEL_SH][SOBEL_SW];
    int tx=threadIdx.x,ty=threadIdx.y;
    int x=blockIdx.x*SOBEL_BW+tx, y=blockIdx.y*SOBEL_BH+ty;
    for(int dy=ty;dy<SOBEL_SH;dy+=SOBEL_BH)
        for(int dx=tx;dx<SOBEL_SW;dx+=SOBEL_BW) {
            int gx=blockIdx.x*SOBEL_BW+dx-1, gy=blockIdx.y*SOBEL_BH+dy-1;
            smem[dy][dx]=in[min(max(gy,0),H-1)*W+min(max(gx,0),W-1)];
        }
    __syncthreads();
    if(x>=W||y>=H){return;}
    float p00=smem[ty][tx],p01=smem[ty][tx+1],p02=smem[ty][tx+2];
    float p10=smem[ty+1][tx],                  p12=smem[ty+1][tx+2];
    float p20=smem[ty+2][tx],p21=smem[ty+2][tx+1],p22=smem[ty+2][tx+2];
    float gx=-p00+p02-2.f*p10+2.f*p12-p20+p22;
    float gy=-p00-2.f*p01-p02+p20+2.f*p21+p22;
    float mag=sqrtf(gx*gx+gy*gy);
    if(x>0&&x<W-1&&y>0&&y<H-1){edges[y*W+x]=mag;edge_u8[y*W+x]=(mag>thr)?255:0;}
    else{edges[y*W+x]=0.f;edge_u8[y*W+x]=0;}
}

/* ── CPU references ──────────────────────────────────────────────── */
static void cpu_rgb2gray(const unsigned char*rgb,float*g,int W,int H){
    for(int i=0;i<W*H;++i) g[i]=0.299f*rgb[i*3]+0.587f*rgb[i*3+1]+0.114f*rgb[i*3+2];}
static void cpu_blur(const float*in,float*out,int W,int H){
    static const float k[5][5]={
        {0.00296902f,0.01330621f,0.02193823f,0.01330621f,0.00296902f},
        {0.01330621f,0.05963430f,0.09832033f,0.05963430f,0.01330621f},
        {0.02193823f,0.09832033f,0.16210282f,0.09832033f,0.02193823f},
        {0.01330621f,0.05963430f,0.09832033f,0.05963430f,0.01330621f},
        {0.00296902f,0.01330621f,0.02193823f,0.01330621f,0.00296902f}};
    for(int y=0;y<H;++y) for(int x=0;x<W;++x){
        float a=0.f;
        for(int ky=-2;ky<=2;++ky) for(int kx=-2;kx<=2;++kx)
            a+=k[ky+2][kx+2]*in[(y+ky<0?0:y+ky>=H?H-1:y+ky)*W+(x+kx<0?0:x+kx>=W?W-1:x+kx)];
        out[y*W+x]=a;}}
static void cpu_sobel(const float*in,unsigned char*e,int W,int H,float thr){
    for(int y=1;y<H-1;++y) for(int x=1;x<W-1;++x){
        float p00=in[(y-1)*W+(x-1)],p01=in[(y-1)*W+x],p02=in[(y-1)*W+(x+1)];
        float p10=in[y*W+(x-1)],p12=in[y*W+(x+1)];
        float p20=in[(y+1)*W+(x-1)],p21=in[(y+1)*W+x],p22=in[(y+1)*W+(x+1)];
        float gx=-p00+p02-2.f*p10+2.f*p12-p20+p22;
        float gy=-p00-2.f*p01-p02+p20+2.f*p21+p22;
        e[y*W+x]=(sqrtf(gx*gx+gy*gy)>thr)?255:0;}}

static float gpu_ms(cudaEvent_t s,cudaEvent_t e){
    float ms;cudaEventSynchronize(e);cudaEventElapsedTime(&ms,s,e);return ms;}

static void do_bench(int W,int H,cudaEvent_t evs,cudaEvent_t eve,FILE*csv){
    size_t npix=W*H;
    unsigned char*h_rgb=(unsigned char*)malloc(npix*3);
    float*h_g=(float*)malloc(npix*4), *h_b=(float*)malloc(npix*4),
          *h_br=(float*)malloc(npix*4);
    unsigned char*h_e=(unsigned char*)malloc(npix),*h_er=(unsigned char*)malloc(npix);
    /* synthetic test image */
    for(int y=0;y<H;++y) for(int x=0;x<W;++x){
        int p=(y*W+x)*3;
        h_rgb[p]=(unsigned char)(255*x/W);
        h_rgb[p+1]=(unsigned char)(255*y/H);
        h_rgb[p+2]=(unsigned char)(((x/32+y/32)&1)*255);
    }
    unsigned char*d_rgb; float*d_g,*d_t,*d_b,*d_ef; unsigned char*d_eu;
    cudaMalloc(&d_rgb,npix*3); cudaMalloc(&d_g,npix*4); cudaMalloc(&d_t,npix*4);
    cudaMalloc(&d_b,npix*4);   cudaMalloc(&d_ef,npix*4); cudaMalloc(&d_eu,npix);
    cudaMemcpy(d_rgb,h_rgb,npix*3,cudaMemcpyHostToDevice);
    dim3 blk(BLOCK_W,BLOCK_H), grd((W+BLOCK_W-1)/BLOCK_W,(H+BLOCK_H-1)/BLOCK_H);
    dim3 sblk(SOBEL_BW,SOBEL_BH), sgrd((W+SOBEL_BW-1)/SOBEL_BW,(H+SOBEL_BH-1)/SOBEL_BH);
    float thr=30.f,ms;
    rgb_to_grayscale<<<grd,blk>>>(d_rgb,d_g,W,H); cudaDeviceSynchronize();

    /* naive */
    for(int i=0;i<3;++i) gaussian_blur_naive<<<grd,blk>>>(d_g,d_b,W,H);
    cudaDeviceSynchronize();
    cudaEventRecord(evs);
    for(int i=0;i<20;++i) gaussian_blur_naive<<<grd,blk>>>(d_g,d_b,W,H);
    cudaEventRecord(eve); float naive_ms=gpu_ms(evs,eve)/20;

    /* separable */
    for(int i=0;i<3;++i){gaussian_blur_horiz<<<grd,blk>>>(d_g,d_t,W,H);gaussian_blur_vert<<<grd,blk>>>(d_t,d_b,W,H);}
    cudaDeviceSynchronize();
    cudaEventRecord(evs);
    for(int i=0;i<20;++i){gaussian_blur_horiz<<<grd,blk>>>(d_g,d_t,W,H);gaussian_blur_vert<<<grd,blk>>>(d_t,d_b,W,H);}
    cudaEventRecord(eve); float sep_ms=gpu_ms(evs,eve)/20;

    /* shmem */
    for(int i=0;i<3;++i) gaussian_blur_shmem<<<grd,blk>>>(d_g,d_b,W,H);
    cudaDeviceSynchronize();
    cudaEventRecord(evs);
    for(int i=0;i<20;++i) gaussian_blur_shmem<<<grd,blk>>>(d_g,d_b,W,H);
    cudaEventRecord(eve); float shmem_ms=gpu_ms(evs,eve)/20;
    cudaMemcpy(h_b,d_b,npix*4,cudaMemcpyDeviceToHost);

    /* sobel */
    for(int i=0;i<3;++i) sobel_edge_shmem<<<sgrd,sblk>>>(d_b,d_ef,d_eu,W,H,thr);
    cudaDeviceSynchronize();
    cudaEventRecord(evs);
    for(int i=0;i<20;++i) sobel_edge_shmem<<<sgrd,sblk>>>(d_b,d_ef,d_eu,W,H,thr);
    cudaEventRecord(eve); float sobel_ms=gpu_ms(evs,eve)/20;
    cudaMemcpy(h_e,d_eu,npix,cudaMemcpyDeviceToHost);

    /* full pipeline */
    cudaEventRecord(evs);
    for(int i=0;i<20;++i){
        rgb_to_grayscale<<<grd,blk>>>(d_rgb,d_g,W,H);
        gaussian_blur_shmem<<<grd,blk>>>(d_g,d_b,W,H);
        sobel_edge_shmem<<<sgrd,sblk>>>(d_b,d_ef,d_eu,W,H,thr);
    }
    cudaEventRecord(eve); float pipe_ms=gpu_ms(evs,eve)/20;

    /* CPU */
    cpu_rgb2gray(h_rgb,h_g,W,H);
    struct timespec t0,t1;
    clock_gettime(CLOCK_MONOTONIC,&t0);
    cpu_blur(h_g,h_br,W,H);
    clock_gettime(CLOCK_MONOTONIC,&t1);
    float cpu_blur_ms=(float)((t1.tv_sec-t0.tv_sec)*1e3+(t1.tv_nsec-t0.tv_nsec)*1e-6);
    clock_gettime(CLOCK_MONOTONIC,&t0);
    cpu_rgb2gray(h_rgb,h_g,W,H); cpu_blur(h_g,h_br,W,H); cpu_sobel(h_br,h_er,W,H,thr);
    clock_gettime(CLOCK_MONOTONIC,&t1);
    float cpu_total_ms=(float)((t1.tv_sec-t0.tv_sec)*1e3+(t1.tv_nsec-t0.tv_nsec)*1e-6);

    printf("%dx%d  naive=%.2f sep=%.2f shmem=%.2f sobel=%.2f pipe=%.2f  cpu_blur=%.2f cpu_total=%.2f  speedup=%.1fx\n",
           W,H,naive_ms,sep_ms,shmem_ms,sobel_ms,pipe_ms,cpu_blur_ms,cpu_total_ms,cpu_total_ms/pipe_ms);
    fprintf(csv,"%d,%d,%.4f,%.4f,%.4f,%.4f,%.4f,%.4f,%.4f\n",
            W,H,naive_ms,sep_ms,shmem_ms,sobel_ms,pipe_ms,cpu_blur_ms,cpu_total_ms);

    cudaFree(d_rgb);cudaFree(d_g);cudaFree(d_t);cudaFree(d_b);cudaFree(d_ef);cudaFree(d_eu);
    free(h_rgb);free(h_g);free(h_b);free(h_br);free(h_e);free(h_er);
}

int main(void){
    cudaDeviceProp p; cudaGetDeviceProperties(&p,0);
    printf("GPU: %s  SMs:%d  Shmem:%.0fKB\n",p.name,p.multiProcessorCount,p.sharedMemPerBlock/1024.f);
    FILE*csv=fopen("img_results.csv","w");
    fprintf(csv,"W,H,naive_ms,sep_ms,shmem_ms,sobel_ms,pipe_ms,cpu_blur_ms,cpu_total_ms\n");
    cudaEvent_t evs,eve; cudaEventCreate(&evs); cudaEventCreate(&eve);
    int sizes[][2]={{640,480},{1280,720},{1920,1080},{3840,2160}};
    for(int i=0;i<4;++i) do_bench(sizes[i][0],sizes[i][1],evs,eve,csv);
    fclose(csv); return 0;
}
'''

with open('image_kernels.cu','w') as f: f.write(cuda_code)
print('Written image_kernels.cu')

In [ ]:
# ── Cell 5: Compile ──────────────────────────────────────────────────
!nvcc -O2 -arch=sm_75 --use_fast_math image_kernels.cu -o img_bench 2>&1
print('Compiled!')

In [ ]:
# ── Cell 6: Run benchmark ────────────────────────────────────────────
!./img_bench

In [ ]:
# ── Cell 7: GPU image processing in Python (for visualisation) ───────
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
import time
import cv2

# ── Load and resize image for demo ──────────────────────────────────
img_pil = Image.open('../test_4k.jpg').convert('RGB')
# Use 1920×1080 for visualisation (4K is slow to display)
img_1080 = img_pil.resize((1920, 1080), Image.LANCZOS)
img_np = np.array(img_1080, dtype=np.uint8)   # [H, W, 3]
print(f'Working image: {img_np.shape[1]}×{img_np.shape[0]}')

# ── GPU pipeline using PyTorch (shows how our CUDA extension would plug in) ─
def gpu_pipeline_pytorch(img_np):
    """Full pipeline on GPU using PyTorch ops — same algorithm as our CUDA kernels."""
    H, W = img_np.shape[:2]

    # RGB → grayscale
    weights = torch.tensor([0.299, 0.587, 0.114], device='cuda')
    img_t = torch.from_numpy(img_np).float().cuda()  # [H, W, 3]
    gray = (img_t * weights).sum(dim=-1)              # [H, W]

    # Gaussian 5×5 (σ=1.0)
    gauss_kernel = torch.tensor([
        [0.00296902, 0.01330621, 0.02193823, 0.01330621, 0.00296902],
        [0.01330621, 0.05963430, 0.09832033, 0.05963430, 0.01330621],
        [0.02193823, 0.09832033, 0.16210282, 0.09832033, 0.02193823],
        [0.01330621, 0.05963430, 0.09832033, 0.05963430, 0.01330621],
        [0.00296902, 0.01330621, 0.02193823, 0.01330621, 0.00296902]
    ], dtype=torch.float32, device='cuda').view(1, 1, 5, 5)

    gray4d = gray.unsqueeze(0).unsqueeze(0)           # [1, 1, H, W]
    blurred = F.conv2d(gray4d, gauss_kernel, padding=2).squeeze()  # [H, W]

    # Sobel
    sobel_x = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]],
                            dtype=torch.float32, device='cuda').view(1,1,3,3)
    sobel_y = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]],
                            dtype=torch.float32, device='cuda').view(1,1,3,3)
    blur4d = blurred.unsqueeze(0).unsqueeze(0)
    gx = F.conv2d(blur4d, sobel_x, padding=1).squeeze()
    gy = F.conv2d(blur4d, sobel_y, padding=1).squeeze()
    magnitude = torch.sqrt(gx**2 + gy**2)
    edges = (magnitude > 30).byte() * 255

    return (gray.cpu().numpy().astype(np.uint8),
            blurred.cpu().numpy().clip(0, 255).astype(np.uint8),
            edges.cpu().numpy())

# Run GPU pipeline
torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(5): gray_gpu, blur_gpu, edges_gpu = gpu_pipeline_pytorch(img_np)
torch.cuda.synchronize()
gpu_ms = (time.perf_counter() - t0) * 1000 / 5

# CPU pipeline (using cv2 for fair comparison)
t0 = time.perf_counter()
for _ in range(5):
    gray_cpu = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    blur_cpu  = cv2.GaussianBlur(gray_cpu, (5,5), 1.0)
    edges_cpu = cv2.Canny(blur_cpu, 30, 90)
cpu_ms = (time.perf_counter() - t0) * 1000 / 5

print(f'\nPipeline timing on 1920×1080:')
print(f'  GPU (PyTorch): {gpu_ms:.1f} ms')
print(f'  CPU (OpenCV) : {cpu_ms:.1f} ms')
print(f'  Speedup      : {cpu_ms/gpu_ms:.1f}x')

In [ ]:
# ── Cell 8: THE VISUAL DEMO — side-by-side comparison ────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# Use a crop for cleaner visualisation
crop_h, crop_w = 540, 960   # half of 1080p
y0, x0 = 270, 480           # centre crop
orig_crop    = img_np[y0:y0+crop_h, x0:x0+crop_w]
gray_crop    = gray_gpu[y0:y0+crop_h, x0:x0+crop_w]
blur_crop    = blur_gpu[y0:y0+crop_h, x0:x0+crop_w]
edges_g_crop = edges_gpu[y0:y0+crop_h, x0:x0+crop_w]
edges_c_crop = edges_cpu[y0:y0+crop_h, x0:x0+crop_w]

fig = plt.figure(figsize=(20, 12), facecolor='#0d1117')
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.08, wspace=0.04)

panels = [
    (0, 0, orig_crop,    'Original (RGB)',             'viridis', False),
    (0, 1, gray_crop,    'Step 1: Grayscale (GPU)',    'gray',    False),
    (0, 2, blur_crop,    'Step 2: Gaussian Blur (GPU)','gray',    False),
    (1, 0, edges_g_crop, 'Step 3: Sobel Edges (GPU)',  'gray',    False),
    (1, 1, edges_c_crop, 'Step 3: Sobel Edges (CPU)',  'gray',    False),
    (1, 2, np.abs(edges_g_crop.astype(int) - edges_c_crop.astype(int)).astype(np.uint8),
                         'Diff: GPU vs CPU',            'hot',     False),
]

for row, col, data, title, cmap, _ in panels:
    ax = fig.add_subplot(gs[row, col])
    ax.imshow(data, cmap=cmap if data.ndim == 2 else None, vmin=0, vmax=255)
    ax.set_title(title, color='white', fontsize=13, fontweight='bold', pad=8)
    ax.axis('off')

fig.suptitle(
    f'GPU Image Processing Pipeline  |  1920×1080  |  '
    f'GPU: {gpu_ms:.1f}ms  CPU: {cpu_ms:.1f}ms  Speedup: {cpu_ms/gpu_ms:.1f}×',
    color='white', fontsize=15, fontweight='bold', y=0.98
)

plt.savefig('pipeline_demo.png', dpi=120, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print('Saved pipeline_demo.png — this goes in your README!')

In [ ]:
# ── Cell 9: Benchmark charts ─────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

df = pd.read_csv('img_results.csv')
df['megapixels'] = df['W'] * df['H'] / 1e6
df['speedup_blur']  = df['cpu_blur_ms']  / df['shmem_ms']
df['speedup_total'] = df['cpu_total_ms'] / df['pipe_ms']
df['label'] = df.apply(lambda r: f"{int(r.W)}×{int(r.H)}", axis=1)

print(df[['label','naive_ms','sep_ms','shmem_ms','sobel_ms','pipe_ms',
          'cpu_blur_ms','cpu_total_ms','speedup_blur','speedup_total']].to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.patch.set_facecolor('white')
x = np.arange(len(df))

# ── Plot 1: Blur kernel comparison ───────────────────────────────────
ax = axes[0]
w = 0.22
ax.bar(x - w,   df['naive_ms'],  w, label='Naive 2D',    color='#e07b39', alpha=0.85)
ax.bar(x,        df['sep_ms'],   w, label='Separable',   color='#888888', alpha=0.85)
ax.bar(x + w,    df['shmem_ms'], w, label='Shmem tiled', color='#2a7abf', alpha=0.85)
ax.bar(x + 2*w,  df['cpu_blur_ms'], w, label='CPU ref', color='#cc3333', alpha=0.85)
ax.set_xticks(x + 0.5*w); ax.set_xticklabels(df['label'], fontsize=9)
ax.set_ylabel('Time (ms)'); ax.set_yscale('log')
ax.set_title('Gaussian blur kernels', fontweight='bold')
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

# ── Plot 2: Speedup vs CPU ────────────────────────────────────────────
ax2 = axes[1]
bars = ax2.bar(x, df['speedup_total'], 0.5, color='#1a9e5c', alpha=0.85)
ax2.axhline(1.0, color='#888', linestyle='--', linewidth=1.2)
for bar, su in zip(bars, df['speedup_total']):
    ax2.text(bar.get_x()+bar.get_width()/2, su+0.3, f'{su:.0f}×',
             ha='center', fontsize=10, fontweight='bold', color='#1a9e5c')
ax2.set_xticks(x); ax2.set_xticklabels(df['label'], fontsize=9)
ax2.set_ylabel('Speedup over CPU (×)'); ax2.set_title('Full pipeline speedup', fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# ── Plot 3: Throughput (megapixels/sec) ───────────────────────────────
ax3 = axes[2]
gpu_mpps = df['megapixels'] / (df['pipe_ms'] / 1000)
cpu_mpps = df['megapixels'] / (df['cpu_total_ms'] / 1000)
ax3.plot(df['megapixels'], gpu_mpps, 'o-', color='#2a7abf', linewidth=2,
         markersize=8, label='GPU pipeline')
ax3.plot(df['megapixels'], cpu_mpps, 's-', color='#cc3333', linewidth=2,
         markersize=8, label='CPU pipeline')
ax3.set_xlabel('Image size (Megapixels)')
ax3.set_ylabel('Throughput (MP/sec)')
ax3.set_title('Throughput vs image size', fontweight='bold')
ax3.legend(); ax3.grid(alpha=0.3)
for mp, gp, cp in zip(df['megapixels'], gpu_mpps, cpu_mpps):
    ax3.annotate(f'{gp:.0f}', (mp, gp), textcoords='offset points',
                 xytext=(5, 5), fontsize=8, color='#2a7abf')

plt.suptitle('GPU Image Processing Pipeline — Kernel Benchmarks', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('benchmark_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved benchmark_chart.png')

In [ ]:
# ── Cell 10: Memory bandwidth analysis ───────────────────────────────
# Key interview topic: is this kernel compute-bound or memory-bound?
import pandas as pd
import torch

df = pd.read_csv('img_results.csv')

# T4 peak memory bandwidth: ~300 GB/s
T4_PEAK_BW_GBs = 300.0

print('Memory bandwidth analysis (Gaussian blur shmem kernel):')
print(f'{"Size":<12}  {"MP":>5}  {"Shmem ms":>10}  {"Bytes r+w":>12}  {"BW GB/s":>10}  {"% peak":>8}')
print('─' * 65)

for _, row in df.iterrows():
    W, H = int(row.W), int(row.H)
    npix = W * H
    # reads: input image (float32), writes: output image (float32)
    # shmem kernel still reads each pixel ~once (halo reads < 10% overhead)
    bytes_rw = npix * 4 * 2  # read input + write output
    ms = row.shmem_ms
    bw = bytes_rw / (ms * 1e-3) / 1e9  # GB/s
    pct = 100 * bw / T4_PEAK_BW_GBs
    label = f"{W}×{H}"
    print(f'{label:<12}  {npix/1e6:>5.1f}  {ms:>10.3f}  {bytes_rw/1e6:>10.0f}MB  {bw:>10.1f}  {pct:>7.1f}%')

print(f'\nT4 peak memory bandwidth: {T4_PEAK_BW_GBs:.0f} GB/s')
print('If % peak > 70%: kernel is memory-bandwidth bound (good — you are hitting hardware limit)')
print('If % peak < 30%: kernel is compute-bound or has launch overhead')
print('\nThis is a key interview talking point: image processing is ALWAYS memory-bandwidth bound.')

In [ ]:
# ── Cell 11: Tile size sweep — find optimal BLOCK_W ─────────────────
# Shows you understand occupancy tuning
import subprocess, re
import matplotlib.pyplot as plt

tile_configs = [(8,8),(16,8),(32,8),(16,16),(32,16)]
times = []
W, H = 1920, 1080

tile_sweep_code = r'''
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

#define KERNEL_R   2
#define KERNEL_SIZE (2*KERNEL_R+1)
#define BW {bw}
#define BH {bh}
#define SW (BW + 2*KERNEL_R)
#define SH (BH + 2*KERNEL_R)

__constant__ float ck[KERNEL_SIZE][KERNEL_SIZE] = {{
    {0.00296902f,0.01330621f,0.02193823f,0.01330621f,0.00296902f},
    {0.01330621f,0.05963430f,0.09832033f,0.05963430f,0.01330621f},
    {0.02193823f,0.09832033f,0.16210282f,0.09832033f,0.02193823f},
    {0.01330621f,0.05963430f,0.09832033f,0.05963430f,0.01330621f},
    {0.00296902f,0.01330621f,0.02193823f,0.01330621f,0.00296902f}}};

__global__ void blur_tile(const float*in,float*out,int W,int H){{
    __shared__ float s[SH][SW];
    int tx=threadIdx.x,ty=threadIdx.y;
    int x=blockIdx.x*BW+tx,y=blockIdx.y*BH+ty;
    for(int dy=ty;dy<SH;dy+=BH) for(int dx=tx;dx<SW;dx+=BW){{
        int gx=blockIdx.x*BW+dx-KERNEL_R,gy=blockIdx.y*BH+dy-KERNEL_R;
        s[dy][dx]=in[min(max(gy,0),H-1)*W+min(max(gx,0),W-1)];}}
    __syncthreads();
    if(x>=W||y>=H) return;
    float a=0.f;
    for(int ky=0;ky<KERNEL_SIZE;++ky) for(int kx=0;kx<KERNEL_SIZE;++kx)
        a+=ck[ky][kx]*s[ty+ky][tx+kx];
    out[y*W+x]=a;}}

int main(){{
    int W={W},H={H}; size_t n=W*H;
    float*di,*do2; cudaMalloc(&di,n*4); cudaMalloc(&do2,n*4);
    cudaMemset(di,0,n*4);
    dim3 blk(BW,BH),grd((W+BW-1)/BW,(H+BH-1)/BH);
    for(int i=0;i<3;++i) blur_tile<<<grd,blk>>>(di,do2,W,H);
    cudaDeviceSynchronize();
    cudaEvent_t s,e; cudaEventCreate(&s); cudaEventCreate(&e);
    cudaEventRecord(s);
    for(int i=0;i<20;++i) blur_tile<<<grd,blk>>>(di,do2,W,H);
    cudaEventRecord(e); float ms; cudaEventSynchronize(e);
    cudaEventElapsedTime(&ms,s,e); ms/=20;
    printf("%.4f\n",ms);
    cudaFree(di); cudaFree(do2); return 0;}}
'''

labels = []
for bw, bh in tile_configs:
    code = tile_sweep_code.format(bw=bw, bh=bh, W=W, H=H)
    with open('tile_bench.cu', 'w') as f: f.write(code)
    r = subprocess.run(['nvcc','-O2','-arch=sm_75','--use_fast_math',
                        'tile_bench.cu','-o','tile_bench'],
                       capture_output=True, text=True)
    if r.returncode == 0:
        out = subprocess.run(['./tile_bench'], capture_output=True, text=True)
        ms = float(out.stdout.strip())
        times.append(ms)
        labels.append(f'{bw}×{bh}')
        print(f'Block {bw}×{bh}: {ms:.3f} ms  ({W*H/1e6/(ms/1000):.0f} MP/s)')
    else:
        print(f'Block {bw}×{bh}: compile failed')
        times.append(None); labels.append(f'{bw}×{bh}')

# Plot tile sweep
valid = [(l,t) for l,t in zip(labels,times) if t is not None]
if valid:
    ls, ts = zip(*valid)
    best_idx = ts.index(min(ts))
    colors = ['#2a7abf' if i==best_idx else '#888888' for i in range(len(ls))]
    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(ls, ts, color=colors, alpha=0.85)
    bars[best_idx].set_edgecolor('#1a9e5c'); bars[best_idx].set_linewidth(2)
    ax.set_ylabel('Time (ms)'); ax.set_title(f'Block size sweep — {W}×{H} Gaussian blur', fontweight='bold')
    for bar, t in zip(bars, ts):
        ax.text(bar.get_x()+bar.get_width()/2, t+0.01, f'{t:.2f}ms',
                ha='center', fontsize=9)
    ax.set_xlabel('Thread block size (W×H)')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('tile_sweep.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'\nBest block size: {ls[best_idx]}  ({ts[best_idx]:.3f} ms)')
    print('This graph goes in your README — shows you understand occupancy tuning')

In [ ]:
# ── Cell 12: Nsight profiling ─────────────────────────────────────────
!nsys profile --stats=true --output=img_profile ./img_bench 2>&1 | head -60
print('\nFull report: img_profile.nsys-rep (download and open in Nsight Systems)')

In [ ]:
# ── Cell 13: Auto-generate README table ──────────────────────────────
import pandas as pd
df = pd.read_csv('img_results.csv')
df['label']   = df.apply(lambda r: f"{int(r.W)}×{int(r.H)}", axis=1)
df['su_blur']  = (df['cpu_blur_ms']  / df['shmem_ms']).round(1)
df['su_total'] = (df['cpu_total_ms'] / df['pipe_ms']).round(1)

print('Copy this table into your GitHub README:\n')
print('| Image size | Naive blur | Sep. blur | Shmem blur | Sobel | GPU pipeline | CPU pipeline | Speedup |')
print('|------------|-----------|-----------|------------|-------|-------------|-------------|---------|')
for _, r in df.iterrows():
    print(f"| {r.label:<10} | {r.naive_ms:.2f} ms   | {r.sep_ms:.2f} ms  | {r.shmem_ms:.2f} ms    | "
          f"{r.sobel_ms:.2f} ms | {r.pipe_ms:.2f} ms      | {r.cpu_total_ms:.2f} ms     | "
          f"**{r.su_total}×** |")